In [ ]:
%load_ext ipy_pdcache

In [ ]:
import io
from typing import BinaryIO

import networkx as nx
import numpy as np
import onto2nx
import pandas as pd
import zstandard as zstd
from gene_map import GeneMapper
from nxontology import imports as nxo_imports
from tqdm.auto import tqdm

# Parameters

In [ ]:
disgenet_fname = snakemake.input.disgenet_fname
gwascatalog_fname = snakemake.input.gwascatalog_fname
efo_fname = snakemake.input.efo_fname
so_fname = snakemake.input.so_fname
dbsnp_hg19_fname = snakemake.input.dbsnp_hg19_fname

db_out_fname = snakemake.output.db_fname
raw_veps_fname = snakemake.output.raw_veps

gwas_gene_source = snakemake.config["parameters"]["associated_gene_source"]
gwas_odds_ratio_threshold = snakemake.config["parameters"].get(
    "gwas_odds_ratio_threshold"
)
annotation_sources = snakemake.config["annotation_sources"]
snp_filters = snakemake.config["snp_filters"]
dbsnp_hg19_source = snakemake.config["input_files"]["dbsnp_hg19"]

# Load DisGeNET

In [ ]:
df_disgenet = pd.read_table(
    disgenet_fname, usecols=["snpId", "diseaseId", "diseaseName", "source"]
)

df_disgenet["snp_source"] = "disgenet"
df_disgenet["diseaseIdType"] = "UMLS_CUI"

In [ ]:
df_disgenet.head()

# Load GWAS catalog

## Parse input

In [ ]:
df_gwascat = pd.read_table(gwascatalog_fname, low_memory=False)

# df_gwascat = df_gwascat[['SNP_ID_CURRENT', 'MAPPED_TRAIT_URI', 'MAPPED_TRAIT']]
df_gwascat.dropna(
    subset=["SNP_ID_CURRENT", "MAPPED_TRAIT_URI", "MAPPED_TRAIT"], inplace=True
)
df_gwascat.rename(
    columns={
        "SNP_ID_CURRENT": "snpId",
        "MAPPED_TRAIT_URI": "diseaseId",
        "MAPPED_TRAIT": "diseaseName",
    },
    inplace=True,
)

df_gwascat["snpId"] = df_gwascat["snpId"].apply(lambda x: f"rs{x}")
df_gwascat["snp_source"] = "gwas_catalog"

df_gwascat["diseaseId"] = df_gwascat["diseaseId"].str.split(",")
df_gwascat = df_gwascat.explode("diseaseId")

df_gwascat["diseaseId"] = df_gwascat["diseaseId"].apply(lambda x: x.split("/")[-1])
df_gwascat["diseaseIdType"] = df_gwascat["diseaseId"].apply(lambda x: x.split("_")[0])

# convert BETA to odds ratio
df_gwascat["odds_ratio"] = df_gwascat["OR or BETA"].apply(
    lambda x: np.exp(x) if x < 1 else x
)

df_gwascat.head(1)

## Infer associated gene(s)

Possible columns:
* REPORTED GENE(S): gene reported by author
* MAPPED GENE: Gene(s) mapped to the strongest SNP (if SNP is intergenic uses upstream and downstream genes)
* SNP_GENE_IDS: Entrez Gene ID

In [ ]:
df_gwascat[["REPORTED GENE(S)", "MAPPED_GENE", "SNP_GENE_IDS"]].head()

In [ ]:
if gwas_gene_source == "reported":
    # are gene names, must be mapped to ENTREZ
    raw_genes = df_gwascat["REPORTED GENE(S)"].str.split(", ").tolist()

    gene_blacklist = {"intergenic", "NR"}
    cur_genes = [
        g
        for gs in raw_genes
        if not isinstance(gs, float)
        for g in gs
        if g not in gene_blacklist
    ]  # isinstance(gs,float) -> gs==np.nan

    gm = GeneMapper()
    df_map = gm.query(
        id_list=cur_genes, source_id_type="Gene_Name", target_id_type="GeneID"
    )
    name2id = df_map.set_index("ID_from").to_dict()["ID_to"]

    entrez_genes = [
        None if isinstance(gs, float) else [name2id[g] for g in gs if g in name2id]
        for gs in raw_genes
    ]
elif gwas_gene_source == "mapped":
    # are already ENTREZ IDs
    raw_genes = df_gwascat["SNP_GENE_IDS"].str.split(", ").tolist()
    entrez_genes = [None if isinstance(gs, float) else gs for gs in raw_genes]
else:
    raise RuntimeError(f'Invalid gene source: "{gwas_gene_source}"')

In [ ]:
df_gwascat["associated_genes"] = [
    None if gs is None else ",".join(sorted(gs)) for gs in entrez_genes
]
df_gwascat[
    ["REPORTED GENE(S)", "MAPPED_GENE", "SNP_GENE_IDS", "associated_genes"]
].head()

## Select relevant columns

In [ ]:
df_gwascat.columns

In [ ]:
df_gwascat = df_gwascat[
    [
        "diseaseId",
        "snpId",
        "snp_source",
        "diseaseIdType",
        "odds_ratio",
        "associated_genes",
    ]
]
df_gwascat.head()

# Combine sources

In [ ]:
df = pd.concat([df_gwascat])  # df_disgenet,
df.head()

# Load required ontologies

In [ ]:
def process_node_name(name: str) -> str:
    if "/" in name:
        name = name.split("/")[-1]
    if ":" in name:
        name = name.replace(":", "_")
    return name

In [ ]:
def nx_info(graph):
    print(graph.number_of_nodes(), graph.number_of_edges())

In [ ]:
def read_zst_file(file_path: str) -> BinaryIO:
    dctx = zstd.ZstdDecompressor()
    with open(file_path, "rb") as compressed_file:
        decompressed_bytes = dctx.decompress(compressed_file.read())
    return io.BytesIO(decompressed_bytes)

In [ ]:
%%time

efo_ontology = nxo_imports.from_file(
    read_zst_file(efo_fname) if efo_fname.endswith(".zst") else efo_fname
)

efo_graph = efo_ontology.graph
efo_graph.name = "efo"

efo_graph = nx.relabel_nodes(
    efo_graph, {n: process_node_name(n) for n in efo_graph.nodes()}
)

print(nx_info(efo_graph))

In [ ]:
%%time

so_graph = onto2nx.parse_owl(so_fname)
so_graph.name = "so"
print(nx_info(so_graph))

# Label diseases

In [ ]:
efo_label_map = {idx: data["name"] for idx, data in efo_graph.nodes(data=True)}

In [ ]:
df["diseaseLabel"] = df["diseaseId"].map(efo_label_map)

In [ ]:
df.head()

# Cancer classification

In [ ]:
nodes_all = list(efo_graph.nodes())

# find all disease-nodes
disease_node = "EFO_0000408"
nodes_disease = list(nx.descendants(efo_graph, disease_node)) + [
    disease_node
]  # disease subtree (vs traits, ...)

# find all cancer diseases
cancer_node = "MONDO_0004992"
nodes_cancer = list(nx.descendants(efo_graph, cancer_node)) + [
    cancer_node
]  # cancer subtree

# assert nodes_cancer <= nodes_disease
# assert nodes_disease <= nodes_all  # ???
print(
    f"#cancer/#disease/#all: {len(nodes_cancer)}/{len(nodes_disease)}/{len(nodes_all)}"
)

In [ ]:
tmp = []
for disease in tqdm(df["diseaseId"].unique()):
    if disease in nodes_disease:
        tmp.append({"diseaseId": disease, "is_cancer": disease in nodes_cancer})
    else:
        tmp.append({"diseaseId": disease, "is_cancer": np.nan})

df_iscancer = pd.DataFrame(tmp)
df_iscancer.head(5)

# SNP annotations

## Retrieve VEP annotations

Variant consequence ontology: http://www.sequenceontology.org/browser/current_release

Description of variant types: https://www.ensembl.org/info/genome/variation/prediction/predicted_data.html

Raw data: ftp://ftp.ensembl.org/pub/release-98/variation/vep/

In [ ]:
snps = df["snpId"].unique().tolist()
print(f"Retrieving annotations for {len(snps)} SNPs")

In [ ]:
%%time
%%pdcache df_anno_raw $raw_veps_fname

import tempfile
import subprocess
import shutil


def query_hg19_coords(bb_path, snp_list):
    if dbsnp_hg19_source.endswith(".csv"):
        print(f"Loading hg19 coordinates from CSV file: {bb_path}...")
        df_mock = pd.read_csv(bb_path)
        return df_mock[["refsnp_id", "chr_name", "chrom_start"]].copy()

    bigbed_binary = shutil.which("bigBedNamedItems")
    if not bigbed_binary:
        fallback_path = "/opt/homebrew/Caskroom/miniconda/base/bin/bigBedNamedItems"
        if os.path.exists(fallback_path):
            bigbed_binary = fallback_path

    if not bigbed_binary:
        raise RuntimeError(
            "UCSC 'bigBedNamedItems' executable not found on PATH or at "
            "'/opt/homebrew/Caskroom/miniconda/base/bin/bigBedNamedItems'. "
            "Please install the UCSC utility and ensure it is available on your PATH."
        )

    if not os.path.exists(bb_path):
        raise FileNotFoundError(f"Local bigBed file not found: {bb_path}")

    print(
        f"Querying hg19 coordinates from local bigBed: {bb_path} using {bigbed_binary}..."
    )
    with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False) as f:
        tmp_path = f.name
        f.write("\n".join(snp_list) + "\n")

    try:
        result = subprocess.run(
            [bigbed_binary, "-nameFile", bb_path, tmp_path, "stdout"],
            capture_output=True,
            text=True,
            check=True,
        )
    except subprocess.CalledProcessError as e:
        raise RuntimeError(
            f"bigBedNamedItems failed (exit {e.returncode}): {e.stderr[:300]}"
        )
    finally:
        os.unlink(tmp_path)

    rows = []
    for line in result.stdout.splitlines():
        if not line.strip():
            continue
        cols = line.split("\t")
        chrom = cols[0].removeprefix("chr")
        chrom_start = int(cols[1]) + 1
        name = cols[3]
        rows.append({"refsnp_id": name, "chr_name": chrom, "chrom_start": chrom_start})
    return pd.DataFrame(rows)


# 1. Load GWAS Catalog to get hg38 coordinates and consequences
df_gwas_raw = pd.read_table(gwascatalog_fname, low_memory=False)
df_gwas_raw.dropna(subset=["SNP_ID_CURRENT"], inplace=True)
df_gwas_raw["refsnp_id"] = df_gwas_raw["SNP_ID_CURRENT"].apply(
    lambda x: f"rs{str(x).removeprefix('rs')}"
)

df_hg38_raw = df_gwas_raw[["refsnp_id", "CHR_ID", "CHR_POS", "CONTEXT"]].copy()
df_hg38_raw.dropna(subset=["CHR_ID", "CHR_POS", "CONTEXT"], inplace=True)

df_hg38_raw["CHR_ID"] = (
    df_hg38_raw["CHR_ID"].astype(str).str.replace("chr", "", case=False).str.strip()
)
df_hg38_raw = df_hg38_raw[~df_hg38_raw["CHR_ID"].str.contains("_")]
df_hg38_raw["CHR_POS"] = df_hg38_raw["CHR_POS"].astype(int)

df_hg38_raw["CONTEXT"] = (
    df_hg38_raw["CONTEXT"].astype(str).str.replace(" x ", ";").str.split(";")
)
df_hg38 = df_hg38_raw.explode("CONTEXT")
df_hg38["CONTEXT"] = df_hg38["CONTEXT"].str.strip()
df_hg38.rename(
    columns={
        "CHR_ID": "chr_name",
        "CHR_POS": "chrom_start",
        "CONTEXT": "consequence_type_tv",
    },
    inplace=True,
)
df_hg38["ensembl_transcript_stable_id"] = None
df_hg38["genome_assembly"] = "hg38"

# 2. Query hg19 coordinates
df_hg19_coords = query_hg19_coords(dbsnp_hg19_fname, snps)

df_hg19 = df_hg19_coords.merge(
    df_hg38[["refsnp_id", "consequence_type_tv"]], on="refsnp_id", how="inner"
)
df_hg19["ensembl_transcript_stable_id"] = None
df_hg19["genome_assembly"] = "hg19"

df_anno_raw = pd.concat([df_hg19, df_hg38], ignore_index=True)

In [ ]:
df_anno_raw.head()

## Convert annotations to usable format

In [ ]:
df_anno = df_anno_raw.copy()

# processing preparations
df_anno["chr_name"] = df_anno["chr_name"].astype(str)

# remove haplotypes (e.g. CHR_HSCHR6_MHC_COX_CTG1)
df_anno = df_anno[~df_anno["chr_name"].str.contains("_")]

# Map Sequence Ontology ID to standard SO ID strings
so_label_map = {
    data["label"]: idx for idx, data in so_graph.nodes(data=True) if "label" in data
}


def map_so_val_to_id(val):
    if pd.isna(val) or val == "":
        return "SO_0001628"  # intergenic_variant
    val_str = str(val).strip()
    return so_label_map.get(val_str, "SO_0001628")


df_anno["consequence_type_tv"] = df_anno["consequence_type_tv"].apply(map_so_val_to_id)

# select most frequent annotations - optimized vectorised implementation
print("Selecting top annotations per SNP...")
counts = (
    df_anno.groupby(["refsnp_id", "genome_assembly", "consequence_type_tv"])
    .size()
    .reset_index(name="count")
)
counts = counts.sort_values(
    by=["refsnp_id", "genome_assembly", "count", "consequence_type_tv"],
    ascending=[True, True, False, True],
)
top_consequences = counts.drop_duplicates(subset=["refsnp_id", "genome_assembly"])

df_anno = df_anno.merge(
    top_consequences[["refsnp_id", "genome_assembly", "consequence_type_tv"]],
    on=["refsnp_id", "genome_assembly", "consequence_type_tv"],
)
df_anno = df_anno.drop_duplicates(subset=["refsnp_id", "genome_assembly"])

# set column names
df_anno.drop("ensembl_transcript_stable_id", axis=1, inplace=True)
df_anno.rename(
    columns={
        "refsnp_id": "snpId",
        "chr_name": "chromosome",
        "chrom_start": "position",
        "consequence_type_tv": "variant_type",
    },
    inplace=True,
)

In [ ]:
df_anno.head()

## Group variant types

### Read sequence ontology (SO)

In [ ]:
exon_subgraph = list(nx.ancestors(so_graph, "SO_0001791")) + ["SO_0001791"]
intron_subgraph = list(nx.ancestors(so_graph, "SO_0001627")) + ["SO_0001627"]
intergenic_subgraph = list(nx.ancestors(so_graph, "SO_0001628")) + ["SO_0001628"]

### Find ontology labels

In [ ]:
so_label_map = {data["label"]: idx for idx, data in so_graph.nodes(data=True)}

### Classify variants

In [ ]:
def classify_vep(vep_id):
    special_cases = {
        "SO_0001621": "exonic",  # NMD_transcript_variant
        "SO_0001620": "exonic",  # mature_miRNA_variant
        "SO_0001630": "exonic",  # splice_region_variant
        "SO_0001619": "intronic",  # non_coding_transcript_variant
    }

    if vep_id in exon_subgraph:
        assert vep_id not in intron_subgraph and vep_id not in intergenic_subgraph, (
            vep_id
        )
        return "exonic"
    elif vep_id in intron_subgraph:
        assert vep_id not in exon_subgraph and vep_id not in intergenic_subgraph, vep_id
        return "intronic"
    elif vep_id in intergenic_subgraph:
        assert vep_id not in intron_subgraph and vep_id not in exon_subgraph, vep_id
        return "intergenic"
    else:
        return special_cases.get(vep_id, "ambiguous")

In [ ]:
df_anno["variant_group"] = df_anno["variant_type"].apply(classify_vep)
df_anno["variant_group"].value_counts()

In [ ]:
df_anno.head()

## Sanity checks

In [ ]:
# assert that all SNPs have been annotated (TODO: make this rigorous)
# assert set(df_anno['snpId'].tolist()) == set(snps), set(snps) - set(df_anno['snpId'].tolist())
assert df_anno is not None
assert df_anno.shape[0] > 0

In [ ]:
# assert that all variant types have been grouped
assert df_anno["variant_group"].isna().sum() == 0, (
    df_anno[df_anno.variant_group.isna()]
    .drop_duplicates("variant_type")["variant_type"]
    .tolist()
)

In [ ]:
# assert that variant type groups are reasonable
# assert set(df_anno['variant_group']) <= {'exonic', 'intronic', 'intergenic', 'ambiguous'}, df_anno['variant_group'].unique().tolist()

In [ ]:
# statistics
print("#SNPs in database:", df["snpId"].nunique(), f"({len(snps)})")
print("#annotated SNPs:", df_anno["snpId"].nunique())
print("#intersection:", len(set(df["snpId"].tolist()) & set(df_anno["snpId"].tolist())))

## Transform dataset

In [ ]:
def dummy_agg(x):
    assert len(x) <= 1
    return x


df_anno_trans = pd.pivot_table(
    df_anno,
    values=["chromosome", "position", "variant_type", "variant_group"],
    index=["snpId"],
    columns=["genome_assembly"],
    aggfunc=dummy_agg,
).reset_index()

df_anno_trans.columns = [
    "_".join(col).rstrip("_") for col in df_anno_trans.columns.values
]

# Copy hg38 annotations to hg19 columns to ensure consistency
df_anno_trans["variant_type_hg19"] = df_anno_trans["variant_type_hg38"]
df_anno_trans["variant_group_hg19"] = df_anno_trans["variant_group_hg38"]

In [ ]:
df_anno_trans.head()

# Merge data sources

In [ ]:
# initial aggregation
df_final = df.copy()
df_final.shape

In [ ]:
# cancer-classification
df_final = df_final.merge(df_iscancer, on="diseaseId")
df_final.shape

In [ ]:
# SNP annotation
df_final = df_final.merge(df_anno_trans, how="left")
df_final.shape

In [ ]:
df_final.head()

# Apply filters

## General filters

In [ ]:
# only keep diseases (and not e.g. traits)
# only keep diseases (and not e.g. traits)
df_final.dropna(subset=["is_cancer"], inplace=True)

# apply Odds Ratio filter
if gwas_odds_ratio_threshold is not None:
    print(f"Filtering by Odds Ratio threshold: {gwas_odds_ratio_threshold}")
    print(f"Before filtering: {df_final.shape[0]} rows")
    df_final = df_final[
        (df_final["odds_ratio"] > gwas_odds_ratio_threshold)
        | (df_final["odds_ratio"] < 1 / gwas_odds_ratio_threshold)
    ]
    print(f"After filtering: {df_final.shape[0]} rows")

In [ ]:
df_final.shape

## Variant type filters (only add marker)

In [ ]:
for filter_name, filter_query in snp_filters.items():
    for genome_assembly in annotation_sources:
        idx = f"filter_{filter_name}_{genome_assembly}"

        df_final[idx] = False
        if filter_query is None:
            df_final[idx] = True
        else:
            match = df_final.query(
                filter_query.format(genome_assembly=genome_assembly)
            ).index
            df_final.loc[match, idx] = True

# Save result

In [ ]:
df_final.head()

In [ ]:
df_final.drop_duplicates(inplace=True)
df_final.to_csv(db_out_fname, index=False)